In [ ]:
%load_ext autoreload
%autoreload 2
import jax
import jax.numpy as jnp
import sys
import os

sys.path.insert(0, os.path.abspath(".."))

from src.training import (
    train_resnet, loss_fn_resnet, loss_fn_fa_resnet,
    loss_fn_ea_resnet, loss_fn_da_resnet, DEFAULT_CONFIG_RESNET,
)
from src.teacher import make_wi_particles_resnet, make_arbitrary_particles_resnet
from src.metrics import rmd_to_projected
from src.utils import plot_rmd_curves, plot_particles_3d_resnet
import src.spaces as spaces

spaces.SETUP = spaces.make_setup_matrix()

L_VALUES = [1, 10, 100, 500]  # ResNet depth (1 skip connection)
N = 5
#N_VALUES = [5, 10, 50, 100, 500]
N_REPS = 3
CONFIG = {**DEFAULT_CONFIG_RESNET, "T": 2.0, "gr": 5}
CONFIG['beta'] = 0.0


In [21]:
teacher_wi = make_wi_particles_resnet(1)

results_wi = {"vanilla": [], "DA": [], "FA": [], "EA": []}

for L in L_VALUES:
    for name in results_wi:
        results_wi[name].append([])

    for rep in range(N_REPS):
        key = jax.random.PRNGKey(rep * 1000 + N)

        # Vanilla
        h = train_resnet(teacher_wi, N, L, jax.random.fold_in(key, 0), CONFIG, "si")
        results_wi["vanilla"][-1].append(rmd_to_projected(h["particles"][-1][0]))

        # DA
        h = train_resnet(teacher_wi, N, L, jax.random.fold_in(key, 1), CONFIG, "si", used_loss_fn=loss_fn_da_resnet)
        results_wi["DA"][-1].append(rmd_to_projected(h["particles"][-1][0]))

        # FA
        h = train_resnet(teacher_wi, N, L, jax.random.fold_in(key, 2), CONFIG, "si", used_loss_fn=loss_fn_fa_resnet)
        results_wi["FA"][-1].append(rmd_to_projected(h["particles"][-1][0]))

        # EA
        h = train_resnet(teacher_wi, N, L, jax.random.fold_in(key, 3), CONFIG, "si", used_loss_fn=loss_fn_ea_resnet)
        results_wi["EA"][-1].append(rmd_to_projected(h["particles"][-1][0]))

    print(f"N={N} done")

N=5 done
N=5 done
N=5 done
N=5 done


In [ ]:
plot_rmd_curves(L_VALUES, results_wi, "Number of Particles (M)", "Distance to E^G — WI teacher, SI init (ResNet L=1)")


In [23]:
teacher_arb = make_arbitrary_particles_resnet(1)

results_arb = {"vanilla": [], "DA": [], "FA": [], "EA": []}

for L in L_VALUES:
    for name in results_arb:
        results_arb[name].append([])

    for rep in range(N_REPS):
        key = jax.random.PRNGKey(rep * 1000 + N)

        h = train_resnet(teacher_arb, N, L, jax.random.fold_in(key, 0), CONFIG, "si")
        results_arb["vanilla"][-1].append(rmd_to_projected(h["particles"][-1][0]))

        h = train_resnet(teacher_arb, N, L, jax.random.fold_in(key, 1), CONFIG, "si", used_loss_fn=loss_fn_da_resnet)
        results_arb["DA"][-1].append(rmd_to_projected(h["particles"][-1][0]))

        h = train_resnet(teacher_arb, N, L, jax.random.fold_in(key, 2), CONFIG, "si", used_loss_fn=loss_fn_fa_resnet)
        results_arb["FA"][-1].append(rmd_to_projected(h["particles"][-1][0]))

        h = train_resnet(teacher_arb, N, L, jax.random.fold_in(key, 3), CONFIG, "si", used_loss_fn=loss_fn_ea_resnet)
        results_arb["EA"][-1].append(rmd_to_projected(h["particles"][-1][0]))

    print(f"N={N} done")

N=5 done
N=5 done
N=5 done
N=5 done


In [ ]:
plot_rmd_curves(L_VALUES, results_arb, "Number of Particles (M)", "Distance to E^G — arbitrary teacher, SI init (ResNet L=1)")


In [25]:
teacher = make_wi_particles_resnet(1)
N = 5
L = 500
key = jax.random.PRNGKey(42)

configs = {**DEFAULT_CONFIG_RESNET, "T": 1.0, "gr": 10}

h_vanilla = train_resnet(teacher, N, L, jax.random.fold_in(key, 0), configs, "si")
print("vanilla done")
h_da = train_resnet(teacher, N, L, jax.random.fold_in(key, 1), configs, "si", used_loss_fn=loss_fn_da_resnet)
print("DA done")
h_fa = train_resnet(teacher, N, L, jax.random.fold_in(key, 2), configs, "si", used_loss_fn=loss_fn_fa_resnet)
print("FA done")
h_ea = train_resnet(teacher, N, L, jax.random.fold_in(key, 3), configs, "si", used_loss_fn=loss_fn_ea_resnet)
print("EA done")

vanilla done
DA done
FA done
EA done


In [ ]:
plot_particles_3d_resnet(h_vanilla, teacher, "Student Particles — vanilla (ResNet L=1)")


In [ ]:
plot_particles_3d_resnet(h_da, teacher, "Student Particles — DA (ResNet L=1)")


In [ ]:
plot_particles_3d_resnet(h_fa, teacher, "Student Particles — FA (ResNet L=1)")


In [ ]:
plot_particles_3d_resnet(h_ea, teacher, "Student Particles — EA (ResNet L=1)")


In [13]:
teacher_arb2 = make_arbitrary_particles_resnet(L)

h_vanilla_arb = train_resnet(teacher_arb2, N, L, jax.random.fold_in(key, 0), configs, "si")
print("vanilla done")
h_da_arb = train_resnet(teacher_arb2, N, L, jax.random.fold_in(key, 1), configs, "si", used_loss_fn=loss_fn_da_resnet)
print("DA done")
h_fa_arb = train_resnet(teacher_arb2, N, L, jax.random.fold_in(key, 2), configs, "si", used_loss_fn=loss_fn_fa_resnet)
print("FA done")
h_ea_arb = train_resnet(teacher_arb2, N, L, jax.random.fold_in(key, 3), configs, "si", used_loss_fn=loss_fn_ea_resnet)
print("EA done")

vanilla done
DA done
FA done
EA done


In [ ]:
plot_particles_3d_resnet(h_vanilla_arb, teacher_arb2, "Arbitrary teacher — vanilla (ResNet L=1)")


In [ ]:
plot_particles_3d_resnet(h_da_arb, teacher_arb2, "Arbitrary teacher — DA (ResNet L=1)")


In [ ]:
plot_particles_3d_resnet(h_fa_arb, teacher_arb2, "Arbitrary teacher — FA (ResNet L=1)")


In [ ]:
plot_particles_3d_resnet(h_ea_arb, teacher_arb2, "Arbitrary teacher — EA (ResNet L=1)")


In [67]:
import numpy as np

h_vanilla['particles'][-1][0].mean()

Array(-1.6908746, dtype=float32)